# Preprocessing: leak-free split + COCO -> YOLO-seg conversion
Thin wrapper around `src/data/prepare_dataset.py` (the real, runnable pipeline) so the report can show its output inline. See that file's docstring / `README.md` for why the split is grouped by source video id instead of using TrashCan's own train/val split.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
from src.data import prepare_dataset
prepare_dataset.main()

## Inspect the resulting split manifest

In [ ]:
import pandas as pd
from src.config import SPLIT_MANIFEST_PATH
manifest = pd.read_csv(SPLIT_MANIFEST_PATH)
display(manifest.groupby('split')[['n_objects']].agg(['count', 'sum', 'mean']))
manifest.head()

In [ ]:
# Confirm zero video-id leakage between splits (the key correctness check for this step)
for a in ['train', 'val', 'test']:
    for b in ['train', 'val', 'test']:
        if a < b:
            ga = set(manifest.loc[manifest.split == a, 'video_id'].dropna())
            gb = set(manifest.loc[manifest.split == b, 'video_id'].dropna())
            print(a, 'vs', b, '-> shared video ids:', len(ga & gb))